In [1]:
import pandas as pd
import os
import json

# 读取CSV文件
try:
    df = pd.read_csv("车牌数据.csv")
    print(f"✅ 成功读取 {len(df)} 条车牌数据")
    print("\n数据预览：")
    print(df.head(10))
except FileNotFoundError:
    print(f"❌ 找不到文件：车牌数据.csv")
    print("请确认文件在当前目录下")
    raise

# 检查必要的列（支持繁简体）
possible_plate_cols = ['車牌', '车牌', 'plate', 'Plate']
possible_price_cols = ['價格（HK$）', '價格', '价格', 'price', 'Price']

plate_col = None
price_col = None

for col in possible_plate_cols:
    if col in df.columns:
        plate_col = col
        break

for col in possible_price_cols:
    if col in df.columns:
        price_col = col
        break

if plate_col is None:
    raise ValueError(f"找不到『車牌』欄位，現有欄位：{list(df.columns)}")
if price_col is None:
    raise ValueError(f"找不到『價格』欄位，現有欄位：{list(df.columns)}")

print(f"\n✅ 识别到字段：车牌='{plate_col}'，价格='{price_col}'")

# 处理价格列（处理空值和逗号）
def parse_price(price):
    """将价格转换为数字（港元）"""
    # 处理空值（NaN 或 None）
    if price is None or (isinstance(price, float) and pd.isna(price)):
        return 0
    
    if isinstance(price, (int, float)):
        if pd.isna(price):
            return 0
        return int(price)
    
    # 转为字符串处理
    price_str = str(price).strip()
    
    # 如果是空字符串
    if price_str == '' or price_str == 'nan' or price_str == 'NaN':
        return 0
    
    # 去除逗号
    price_str = price_str.replace(',', '')
    
    # 处理 "2000万" 格式
    if '萬' in price_str or '万' in price_str:
        num = float(price_str.replace('萬', '').replace('万', ''))
        return int(num * 10000)
    
    # 处理纯数字
    try:
        return int(float(price_str))
    except:
        print(f"⚠️ 警告：无法解析价格格式：{price}，设为0")
        return 0

# 应用价格解析
df['價格_數字'] = df[price_col].apply(parse_price)

# 过滤掉价格为0的无效数据
valid_df = df[df['價格_數字'] > 0].reset_index(drop=True)

invalid_count = len(df) - len(valid_df)
if invalid_count > 0:
    print(f"\n⚠️ 已过滤掉 {invalid_count} 条无效数据（价格为空或格式错误）")

df = valid_df
print(f"✅ 有效数据笔数：{len(df)}")

if len(df) == 0:
    raise ValueError("没有有效的价格数据，请检查CSV文件格式")

# 生成用于显示的格式化价格字符串
def format_price_display(price_num):
    if price_num >= 10000000:
        return f"{price_num // 1000000}千万港元"
    elif price_num >= 10000:
        return f"{price_num // 10000}万港元"
    else:
        return f"{price_num:,}港元"

df['價格_顯示'] = df['價格_數字'].apply(format_price_display)

# 生成JavaScript数据（不再包含日期字段）
js_data = []
for _, row in df.iterrows():
    js_data.append({
        "plate": str(row[plate_col]).strip(),
        "price": int(row['價格_數字']),
        "priceDisplay": row['價格_顯示']
    })

print(f"\n✅ 已生成 {len(js_data)} 条游戏数据")
print("\n前5条数据示例：")
for item in js_data[:5]:
    print(f"  车牌: {item['plate']}, 价格: {item['priceDisplay']}")

# 将数据转为JSON字符串
data_json = json.dumps(js_data, ensure_ascii=False)

# 生成HTML游戏代码（繁体中文版，无日期栏）
html_code = f'''
<!DOCTYPE html>
<html lang="zh-HK">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>猜車牌價格小遊戲</title>
    <style>
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }}
        
        body {{
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif;
            background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);
            min-height: 100vh;
            display: flex;
            justify-content: center;
            align-items: center;
            padding: 20px;
        }}
        
        .game-container {{
            max-width: 600px;
            width: 100%;
            background: white;
            border-radius: 24px;
            box-shadow: 0 20px 60px rgba(0,0,0,0.3);
            overflow: hidden;
        }}
        
        .game-header {{
            background: linear-gradient(135deg, #f5af19 0%, #f12711 100%);
            padding: 20px;
            text-align: center;
            color: white;
        }}
        
        .game-header h1 {{
            font-size: 24px;
            margin-bottom: 8px;
        }}
        
        .game-header p {{
            font-size: 14px;
            opacity: 0.9;
        }}
        
        .score-area {{
            background: #2c3e50;
            padding: 12px 20px;
            display: flex;
            justify-content: space-between;
            color: white;
            font-weight: bold;
        }}
        
        .score-area span:first-child {{
            font-size: 14px;
        }}
        
        .score-area span:last-child {{
            font-size: 20px;
            color: #f5af19;
        }}
        
        .question-area {{
            padding: 30px 20px 20px;
            text-align: center;
        }}
        
        .license-plate {{
            background: #1a1a2e;
            border-radius: 16px;
            padding: 30px 20px;
            margin-bottom: 25px;
            box-shadow: 0 8px 20px rgba(0,0,0,0.2);
        }}
        
        .plate-text {{
            font-size: 56px;
            font-weight: 900;
            letter-spacing: 4px;
            color: #f5af19;
            text-shadow: 2px 2px 0 #f12711;
            font-family: monospace;
            word-break: break-all;
        }}
        
        .question-text {{
            font-size: 18px;
            color: #555;
            margin-bottom: 20px;
        }}
        
        .options {{
            display: flex;
            flex-direction: column;
            gap: 12px;
            margin-bottom: 20px;
        }}
        
        .option-btn {{
            background: #f8f9fa;
            border: 2px solid #e9ecef;
            border-radius: 12px;
            padding: 14px 20px;
            font-size: 16px;
            font-weight: 600;
            cursor: pointer;
            transition: all 0.3s ease;
            text-align: left;
            display: flex;
            justify-content: space-between;
            align-items: center;
        }}
        
        .option-btn:hover:not(:disabled) {{
            background: #e9ecef;
            transform: translateX(5px);
        }}
        
        .option-btn:disabled {{
            cursor: not-allowed;
            opacity: 0.7;
        }}
        
        .option-btn.correct {{
            background: #d4edda;
            border-color: #28a745;
            color: #155724;
        }}
        
        .option-btn.wrong {{
            background: #f8d7da;
            border-color: #dc3545;
            color: #721c24;
        }}
        
        .option-prefix {{
            font-weight: bold;
            color: #666;
        }}
        
        .result-area {{
            background: #f0f4f8;
            border-radius: 12px;
            padding: 15px;
            margin: 15px 0;
            text-align: left;
            display: none;
        }}
        
        .result-area.show {{
            display: block;
            animation: fadeIn 0.5s ease;
        }}
        
        .result-correct {{
            color: #28a745;
            font-weight: bold;
            margin-bottom: 10px;
        }}
        
        .result-wrong {{
            color: #dc3545;
            font-weight: bold;
            margin-bottom: 10px;
        }}
        
        .result-price {{
            font-size: 18px;
            font-weight: bold;
            color: #1a1a2e;
            margin: 10px 0;
        }}
        
        .next-btn {{
            background: linear-gradient(135deg, #f5af19, #f12711);
            border: none;
            border-radius: 40px;
            padding: 14px 30px;
            font-size: 16px;
            font-weight: bold;
            color: white;
            cursor: pointer;
            width: 100%;
            transition: transform 0.3s ease;
            display: none;
        }}
        
        .next-btn.show {{
            display: block;
            animation: fadeIn 0.5s ease;
        }}
        
        .next-btn:hover {{
            transform: translateY(-2px);
        }}
        
        .game-status {{
            text-align: center;
            padding: 20px;
            font-size: 14px;
            color: #999;
            border-top: 1px solid #eee;
        }}
        
        .feedback-icon {{
            font-size: 20px;
            margin-right: 10px;
        }}
        
        @keyframes fadeIn {{
            from {{
                opacity: 0;
                transform: translateY(10px);
            }}
            to {{
                opacity: 1;
                transform: translateY(0);
            }}
        }}
    </style>
</head>
<body>
    <div class="game-container">
        <div class="game-header">
            <h1>🚗 猜猜呢個車牌幾多錢？</h1>
            <p>香港特殊車牌拍賣價格挑戰</p>
        </div>
        
        <div class="score-area">
            <span>📊 進度</span>
            <span id="progress">第 1 / 0 題</span>
        </div>
        
        <div class="question-area">
            <div class="license-plate">
                <div class="plate-text" id="licensePlate">---</div>
            </div>
            <div class="question-text">
                🤔 呢個車牌喺拍賣會上成交價係幾多？
            </div>
            
            <div class="options" id="optionsContainer">
            </div>
            
            <div class="result-area" id="resultArea">
                <div id="resultMessage"></div>
            </div>
            
            <button class="next-btn" id="nextBtn">下一題 →</button>
        </div>
        
        <div class="game-status" id="gameStatus">
            共 <span id="totalCount">0</span> 個車牌等你估！
        </div>
    </div>

    <script>
        const licenseData = {data_json};
        
        let currentIndex = 0;
        let shuffledData = [];
        let answered = false;
        let currentQuestion = null;
        let currentOptions = [];
        
        function formatPriceRange(price) {{
            if (price >= 10000000) {{
                return (price / 1000000).toFixed(0) + "千萬";
            }} else if (price >= 10000) {{
                return (price / 10000).toFixed(0) + "萬";
            }}
            return price.toLocaleString() + "元";
        }}
        
        function generateOptions(correctPrice) {{
            let wrongPrice;
            const randomFactor = Math.random();
            
            if (randomFactor < 0.33) {{
                wrongPrice = correctPrice * (1.3 + Math.random() * 0.5);
            }} else if (randomFactor < 0.66) {{
                wrongPrice = correctPrice * (0.2 + Math.random() * 0.5);
            }} else {{
                wrongPrice = correctPrice * (Math.random() > 0.5 ? 2.5 : 0.4);
            }}
            
            wrongPrice = Math.round(wrongPrice / 10000) * 10000;
            if (wrongPrice < 1000) wrongPrice = 1000;
            
            const options = [
                {{ price: correctPrice, isCorrect: true }},
                {{ price: wrongPrice, isCorrect: false }}
            ];
            
            return options.sort(() => Math.random() - 0.5);
        }}
        
        function getPriceDisplay(price) {{
            if (price >= 10000000) {{
                return (price / 1000000).toFixed(0) + "千萬港元";
            }} else if (price >= 10000) {{
                return (price / 10000).toFixed(0) + "萬港元";
            }}
            return price.toLocaleString() + "港元";
        }}
        
        function getOriginalPriceDisplay(question) {{
            if (question.priceDisplay) {{
                return question.priceDisplay;
            }}
            return getPriceDisplay(question.price);
        }}
        
        function renderQuestion() {{
            if (currentIndex >= shuffledData.length) {{
                document.getElementById("licensePlate").textContent = "🎉 完成！";
                document.querySelector(".question-text").innerHTML = "恭喜你完成咗所有題目！";
                document.getElementById("optionsContainer").innerHTML = "";
                document.getElementById("nextBtn").classList.remove("show");
                document.getElementById("gameStatus").innerHTML = "🎊 好犀利！你已經係車牌拍賣專家啦！ 🎊";
                return;
            }}
            
            currentQuestion = shuffledData[currentIndex];
            document.getElementById("licensePlate").textContent = currentQuestion.plate;
            document.getElementById("progress").textContent = `第 ${{currentIndex + 1}} / ${{shuffledData.length}} 題`;
            
            currentOptions = generateOptions(currentQuestion.price);
            
            const optionsContainer = document.getElementById("optionsContainer");
            optionsContainer.innerHTML = "";
            
            currentOptions.forEach((option, idx) => {{
                const btn = document.createElement("button");
                btn.className = "option-btn";
                btn.innerHTML = `
                    <span class="option-prefix">${{String.fromCharCode(65 + idx)}}.</span>
                    <span>${{getPriceDisplay(option.price)}}</span>
                `;
                btn.onclick = () => checkAnswer(option.isCorrect, option.price);
                optionsContainer.appendChild(btn);
            }});
            
            answered = false;
            document.getElementById("resultArea").classList.remove("show");
            document.getElementById("nextBtn").classList.remove("show");
            
            document.querySelectorAll(".option-btn").forEach(btn => {{
                btn.disabled = false;
                btn.classList.remove("correct", "wrong");
            }});
        }}
        
        function checkAnswer(isCorrect, selectedPrice) {{
            if (answered) return;
            answered = true;
            
            const resultArea = document.getElementById("resultArea");
            const resultMessage = document.getElementById("resultMessage");
            
            document.querySelectorAll(".option-btn").forEach(btn => {{
                btn.disabled = true;
            }});
            
            document.querySelectorAll(".option-btn").forEach((btn, idx) => {{
                const optionPrice = currentOptions[idx].price;
                if (optionPrice === currentQuestion.price) {{
                    btn.classList.add("correct");
                }} else if (optionPrice === selectedPrice && !isCorrect) {{
                    btn.classList.add("wrong");
                }}
            }});
            
            const correctDisplay = getOriginalPriceDisplay(currentQuestion);
            const selectedDisplay = getPriceDisplay(selectedPrice);
            
            if (isCorrect) {{
                resultMessage.innerHTML = `
                    <div class="result-correct">
                        <span class="feedback-icon">✅</span> 恭喜！答啱咗！
                    </div>
                    <div class="result-price">
                        💰 成交價：${{correctDisplay}}
                    </div>
                `;
            }} else {{
                resultMessage.innerHTML = `
                    <div class="result-wrong">
                        <span class="feedback-icon">❌</span> 唔好意思，答錯咗！
                    </div>
                    <div class="result-price">
                        💰 正確答案：${{correctDisplay}}
                    </div>
                    <div class="result-price">
                        📊 你揀咗：${{selectedDisplay}}
                    </div>
                `;
            }}
            
            resultArea.classList.add("show");
            document.getElementById("nextBtn").classList.add("show");
        }}
        
        function nextQuestion() {{
            if (!answered) return;
            currentIndex++;
            renderQuestion();
        }}
        
        function shuffleArray(arr) {{
            for (let i = arr.length - 1; i > 0; i--) {{
                const j = Math.floor(Math.random() * (i + 1));
                [arr[i], arr[j]] = [arr[j], arr[i]];
            }}
            return arr;
        }}
        
        function initGame() {{
            if (licenseData.length === 0) {{
                document.getElementById("licensePlate").textContent = "⚠️ 無數據";
                document.querySelector(".question-text").innerHTML = "請檢查CSV檔案格式";
                return;
            }}
            shuffledData = shuffleArray([...licenseData]);
            currentIndex = 0;
            answered = false;
            document.getElementById("totalCount").textContent = shuffledData.length;
            renderQuestion();
        }}
        
        document.getElementById("nextBtn").onclick = nextQuestion;
        initGame();
    </script>
</body>
</html>
'''

# 在Jupyter Notebook中显示游戏
from IPython.display import display, HTML
display(HTML(html_code))

# 保存为HTML文件到桌面
output_path = os.path.expanduser("~/Desktop/猜車牌遊戲.html")
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(html_code)
print(f"\n✅ 遊戲已保存至：{output_path}")
print("   雙擊該檔案即可用瀏覽器開啟遊戲")

✅ 成功读取 64 条车牌数据

数据预览：
         車牌    價格（HK$）  Unnamed: 2  Unnamed: 3
0        18  2,100,000         NaN         NaN
1  1 L0VE U  1,400,000         NaN         NaN
2        WE    650,000         NaN         NaN
3      CCUE    460,000         NaN         NaN
4  H 8888 H    145,000         NaN         NaN
5      BEST    125,000         NaN         NaN
6  G00D DAY    120,000         NaN         NaN
7  LOVE1314    120,000         NaN         NaN
8     BE BE    110,000         NaN         NaN
9  BOND 007    105,000         NaN         NaN

✅ 识别到字段：车牌='車牌'，价格='價格（HK$）'

⚠️ 已过滤掉 54 条无效数据（价格为空或格式错误）
✅ 有效数据笔数：10

✅ 已生成 10 条游戏数据

前5条数据示例：
  车牌: 18, 价格: 210万港元
  车牌: 1 L0VE U, 价格: 140万港元
  车牌: WE, 价格: 65万港元
  车牌: CCUE, 价格: 46万港元
  车牌: H 8888 H, 价格: 14万港元



✅ 遊戲已保存至：/Users/yuziqi/Desktop/猜車牌遊戲.html
   雙擊該檔案即可用瀏覽器開啟遊戲
